# Reinforcement Learning – GridWorld Exploration

This notebook walks through training and visualising the Q-learning agent
defined in `../code/rl_agent.py`.

In [ ]:
import sys
sys.path.insert(0, '../code')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from rl_agent import GridWorld, QLearningAgent, train

## 1. Create the environment and agent

In [ ]:
env = GridWorld(rows=5, cols=5, start=(0, 0), goal=(4, 4))
agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
)
print(f'Environment: {env.rows}x{env.cols} grid')
print(f'States: {env.n_states}, Actions: {env.n_actions}')

## 2. Train the agent

In [ ]:
rewards = train(env, agent, n_episodes=500, seed=42)
print(f'Mean reward (last 100 episodes): {rewards[-100:].mean():.3f}')

## 3. Plot the training curve

In [ ]:
window = 50
smoothed = np.convolve(rewards, np.ones(window) / window, mode='valid')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rewards, alpha=0.3, label='Episode reward')
ax.plot(range(window - 1, len(rewards)), smoothed,
        label=f'Moving avg ({window})', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Total reward')
ax.set_title('Q-Learning Training Curve – 5×5 GridWorld')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Visualise the learned Q-table (greedy policy)

In [ ]:
ACTION_SYMBOLS = {0: '↑', 1: '↓', 2: '←', 3: '→'}

print('Greedy policy (best action per cell):')
for r in range(env.rows):
    row_str = ''
    for c in range(env.cols):
        state = r * env.cols + c
        if (r, c) == env.goal:
            row_str += ' G '
        else:
            best_action = int(np.argmax(agent.q_table[state]))
            row_str += f' {ACTION_SYMBOLS[best_action]} '
    print(row_str)

## 5. Hyperparameter experiments

Try changing `alpha`, `gamma`, or `epsilon_decay` below and re-run the training cell to see how performance changes.

In [ ]:
experiments = [
    {'alpha': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995, 'label': 'baseline'},
    {'alpha': 0.5, 'gamma': 0.99, 'epsilon_decay': 0.995, 'label': 'high-alpha'},
    {'alpha': 0.1, 'gamma': 0.90, 'epsilon_decay': 0.995, 'label': 'low-gamma'},
]

fig, ax = plt.subplots(figsize=(9, 4))
for cfg in experiments:
    env_exp = GridWorld()
    agent_exp = QLearningAgent(
        n_states=env_exp.n_states,
        n_actions=env_exp.n_actions,
        alpha=cfg['alpha'],
        gamma=cfg['gamma'],
        epsilon_decay=cfg['epsilon_decay'],
    )
    r = train(env_exp, agent_exp, n_episodes=500, seed=42)
    w = 50
    sm = np.convolve(r, np.ones(w) / w, mode='valid')
    ax.plot(range(w - 1, len(r)), sm, label=cfg['label'])

ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed reward')
ax.set_title('Hyperparameter Comparison')
ax.legend()
plt.tight_layout()
plt.show()